# Embedding and Positional Encoding

Models cannot understand words as we do. They need to be converted to tokens first then to numerical representations (embeddings) which the model can understand. Embedding is the first operation in the Transformer. In the 'Attention is all you need' paper, the embeddings are scaled by the square root of the model dimension (d_model). This is done to prevent the positional information to overpower the token information in the positional encodings

In [ ]:
import torch
import math
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [25]:
class InputEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, d_model: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

## Role of positional encoding

Allows the model to understand the position of tokens in a sequence when processing. transformer has limited capacity to keep track of word order as it processes words in parallel. Attention connections are not directional. ensure differences between A B C and C B A . An identifier encoding the position of an embedding

Add the word vector with the position vector
The positional encodings have the same dimensions as the embeddings so that they can be summed

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model) # create a matrix of zeros with the for all embeddings
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1) # create a 1D tensor from 0 to max_sequence_length
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term) # selects all rows anad the even columns
        pe[:, 1::2] = torch.cos(position * div_term) # selects all rows and the odd columns
        self.register_buffer('pe', pe.unsqueeze(0))
        return pe
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)], self.pe[:, :x.size(1)]


In [19]:
vocab_size = 10
d_model = 4
tokens = torch.tensor([1,2,3])
tokens

tensor([1, 2, 3])

In [47]:
embedding_layer = InputEmbeddings(vocab_size=vocab_size, d_model=d_model)
embeddings =  embedding_layer(tokens)

In [50]:
embeddings.detach().numpy()

array([[-1.4911544 , -1.8033533 , -2.6301236 , -2.0472178 ],
       [ 3.9371963 ,  3.943169  , -3.844906  , -1.2341624 ],
       [-1.2974361 ,  0.03611341,  2.7423203 , -1.7845553 ]],
      dtype=float32)

In [46]:
pe_layer = PositionalEncoding(d_model=d_model, max_seq_length=3)
pe, enc = pe_layer(embeddings)


In [48]:
print('embeddings:', embeddings)
print('positional encodings:', enc)
print('summation:', pe )

embeddings: tensor([[-1.4912, -1.8034, -2.6301, -2.0472],
        [ 3.9372,  3.9432, -3.8449, -1.2342],
        [-1.2974,  0.0361,  2.7423, -1.7846]], grad_fn=<MulBackward0>)
positional encodings: tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])
summation: tensor([[[ 1.4941,  0.6775,  0.9379,  3.2260],
         [-2.9303, -5.1075, -2.4082,  1.3710],
         [-3.1119, -0.1700,  1.8851, -0.6030]]], grad_fn=<AddBackward0>)


In [55]:
pe = pe.detach().numpy()

In [ ]:
from ipywidgets import interact
import ipywidgets as widgets

In [ ]:
@interact(d_model=widgets.IntSlider(min=8, max=40, step=8, value=32), 
max_seq_length=widgets.IntSlider(min=8, max=50, step=4, value=50))
def plot_pe(d_model, max_seq_length):
    pe = pe_layer(d_model, max_seq_length)
    plt.imshow(pe.T, aspect='auto', cmap='RdBu')
    plt.colorbar()
    plt.xlabel('position')
    plt.ylabel('dimension')
    plt.show()